In [3]:
!pip install -q pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("taxi").getOrCreate()

df = spark.read.parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet")

df2 = df.filter(df.fare_amount > 0)
df2.count()



UnsupportedOperationException: [FAILED_READ_FILE.UNSUPPORTED_FILE_SYSTEM] Encountered error while reading file https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet. The file system org.apache.hadoop.fs.http.HttpsFileSystem hasn't implemented listStatus. SQLSTATE: KD001

# 新段落

In [ ]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet

df = spark.read.parquet("yellow_tripdata_2021-01.parquet")
df2 = df.filter(df.fare_amount > 0)
df2.count()



In [ ]:
!wget -q "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv" -O zones.csv

zones = spark.read.option("header", True).option("inferSchema", True).csv("zones.csv")
zones.count()

joined = df.join(zones, df.PULocationID == zones.LocationID, "left")
joined.select("PULocationID", "Borough", "Zone", "fare_amount").show(5)


In [4]:
!pip install -q pyspark
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("taxi").getOrCreate()

df = spark.read.parquet("yellow_tripdata_2021-01.parquet")
df.count()


1369769

In [5]:
!wget -q "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv" -O zones.csv

zones = spark.read.option("header", True).option("inferSchema", True).csv("zones.csv")
zones.count()

joined = df.join(zones, df.PULocationID == zones.LocationID, "left")
joined.select("PULocationID", "Borough", "Zone", "fare_amount").show(5)


+------------+---------+--------------------+-----------+
|PULocationID|  Borough|                Zone|fare_amount|
+------------+---------+--------------------+-----------+
|         142|Manhattan| Lincoln Square East|        8.0|
|         238|Manhattan|Upper West Side N...|        3.0|
|         132|   Queens|         JFK Airport|       42.0|
|         138|   Queens|   LaGuardia Airport|       29.0|
|          68|Manhattan|        East Chelsea|       16.5|
+------------+---------+--------------------+-----------+
only showing top 5 rows


In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
joined.explain()

from pyspark.sql.functions import broadcast
joined_b = df.join(broadcast(zones), df.PULocationID == zones.LocationID, "left")
joined_b.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [PULocationID#7L], [cast(LocationID#59 as bigint)], LeftOuter
   :- Sort [PULocationID#7L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(PULocationID#7L, 200), ENSURE_REQUIREMENTS, [plan_id=184]
   :     +- FileScan parquet [VendorID#0L,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3,trip_distance#4,RatecodeID#5,store_and_fwd_flag#6,PULocationID#7L,DOLocationID#8L,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,airport_fee#18] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/yellow_tripdata_2021-01.parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:bigint,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,pas...
   +- Sort [cast(LocationID#59 as bigint) ASC NULLS FIRST], false, 0
      +- 

In [7]:
result = df.groupBy("payment_type").count().orderBy("payment_type")
result.show()


+------------+------+
|payment_type| count|
+------------+------+
|           0| 98352|
|           1|934475|
|           2|322891|
|           3|  8384|
|           4|  5667|
+------------+------+

